In [ ]:
import os
import sys
import json
import argparse
import random
from pathlib import Path

import numpy as np
import nibabel as nib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader


# ============================================================
# Prototype module
# ============================================================

class PrototypeMemory3D(nn.Module):
    def __init__(self, feature_dim=320, num_classes=4, prototypes_per_class=8):
        super().__init__()

        self.feature_dim = feature_dim
        self.num_classes = num_classes
        self.prototypes_per_class = prototypes_per_class
        self.total_prototypes = num_classes * prototypes_per_class

        self.prototypes = nn.Parameter(
            torch.randn(num_classes, prototypes_per_class, feature_dim) * 0.02
        )

    def forward(self, x):
        """
        x: [B, C, D, H, W]
        """

        B, C, D, H, W = x.shape

        x_flat = x.permute(0, 2, 3, 4, 1).reshape(B, D * H * W, C)
        x_norm = F.normalize(x_flat, dim=-1)

        proto_flat = self.prototypes.reshape(self.total_prototypes, C)
        proto_norm = F.normalize(proto_flat, dim=-1)

        sim = torch.matmul(x_norm, proto_norm.t())
        sim = sim.reshape(B, D, H, W, self.num_classes, self.prototypes_per_class)
        sim = sim.permute(0, 4, 5, 1, 2, 3).contiguous()

        class_evidence = sim.max(dim=2).values
        top_proto_indices = sim.reshape(B, self.total_prototypes, D, H, W).argmax(dim=1)

        return {
            "proto_similarity": sim,
            "class_evidence": class_evidence,
            "top_proto_indices": top_proto_indices,
            "prototypes": self.prototypes,
        }


class PrototypeCrossAttention3D(nn.Module):
    def __init__(self, feature_dim=320, num_heads=4):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=feature_dim,
            num_heads=num_heads,
            batch_first=True,
        )

        self.norm = nn.LayerNorm(feature_dim)

    def forward(self, x, prototypes):
        """
        x: [B, C, D, H, W]
        prototypes: [num_classes, prototypes_per_class, C]
        """

        B, C, D, H, W = x.shape

        x_tokens = x.permute(0, 2, 3, 4, 1).reshape(B, D * H * W, C)
        proto_tokens = prototypes.reshape(1, -1, C).repeat(B, 1, 1)

        attended, _ = self.attn(
            query=x_tokens,
            key=proto_tokens,
            value=proto_tokens,
            need_weights=False,
        )

        out = self.norm(x_tokens + attended)
        out = out.reshape(B, D, H, W, C).permute(0, 4, 1, 2, 3).contiguous()

        return out


class PrototypeFusionBlock3D(nn.Module):
    def __init__(
        self,
        feature_dim=320,
        num_classes=4,
        prototypes_per_class=8,
        num_heads=4,
    ):
        super().__init__()

        self.memory = PrototypeMemory3D(
            feature_dim=feature_dim,
            num_classes=num_classes,
            prototypes_per_class=prototypes_per_class,
        )

        self.cross_attention = PrototypeCrossAttention3D(
            feature_dim=feature_dim,
            num_heads=num_heads,
        )

        self.gate = nn.Sequential(
            nn.Conv3d(feature_dim * 2, feature_dim, kernel_size=1),
            nn.Sigmoid(),
        )

        self.out_norm = nn.InstanceNorm3d(feature_dim, affine=True)

    def forward(self, x):
        mem_out = self.memory(x)

        proto_features = self.cross_attention(
            x,
            mem_out["prototypes"],
        )

        gate = self.gate(torch.cat([x, proto_features], dim=1))
        x_fused = gate * proto_features + (1.0 - gate) * x
        x_fused = self.out_norm(x_fused)

        return {
            "x_fused": x_fused,
            "fused_features": x_fused,
            "proto_features": proto_features,
            "gate": gate,
            "proto_similarity": mem_out["proto_similarity"],
            "class_evidence": mem_out["class_evidence"],
            "top_proto_indices": mem_out["top_proto_indices"],
        }


# ============================================================
# Utilities
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def create_5fold_split(case_names, out_json, seed=42):
    case_names = sorted(case_names)

    random.seed(seed)
    random.shuffle(case_names)

    folds = {str(i): [] for i in range(5)}

    for i, case in enumerate(case_names):
        folds[str(i % 5)].append(case)

    split = {}

    for fold in range(5):
        val_cases = folds[str(fold)]
        train_cases = []

        for other_fold in range(5):
            if other_fold != fold:
                train_cases.extend(folds[str(other_fold)])

        split[str(fold)] = {
            "train": sorted(train_cases),
            "val": sorted(val_cases),
        }

    with open(out_json, "w") as f:
        json.dump(split, f, indent=2)

    return split


def dice_score(pred, target, eps=1e-6):
    pred = pred.float()
    target = target.float()

    intersection = (pred * target).sum(dim=(1, 2, 3))
    denominator = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))

    dice = (2 * intersection + eps) / (denominator + eps)
    return dice


def compute_region_dice(logits, target):
    probs = torch.sigmoid(logits)
    pred = (probs > 0.5).float()

    dices = dice_score(pred, target)

    return {
        "WT": dices[0].item(),
        "TC": dices[1].item(),
        "ET": dices[2].item(),
        "Mean": dices.mean().item(),
    }


def unique_params_from_modules(modules):
    seen = set()
    params = []

    for module in modules:
        for p in module.parameters():
            if p.requires_grad and id(p) not in seen:
                params.append(p)
                seen.add(id(p))

    return params


# ============================================================
# Dataset
# ============================================================

class BraTSPatchDataset(Dataset):
    def __init__(self, case_names, data_dir, patch_size=(128, 128, 128)):
        self.case_names = case_names
        self.data_dir = Path(data_dir)
        self.patch_size = patch_size
        self.modalities = ["t1n", "t1c", "t2w", "t2f"]

    def __len__(self):
        return len(self.case_names)

    def _load_nii(self, path):
        arr = nib.load(str(path)).get_fdata().astype(np.float32)
        return arr

    def _normalize(self, x):
        mask = x != 0

        if mask.sum() > 0:
            mean = x[mask].mean()
            std = x[mask].std()
            x = (x - mean) / (std + 1e-8)

        return x

    def _center_crop_or_pad(self, arr, target_shape):
        out = np.zeros(target_shape, dtype=arr.dtype)

        src_slices = []
        dst_slices = []

        for dim, target in enumerate(target_shape):
            size = arr.shape[dim]

            if size >= target:
                start = (size - target) // 2
                src_slices.append(slice(start, start + target))
                dst_slices.append(slice(0, target))
            else:
                start = (target - size) // 2
                src_slices.append(slice(0, size))
                dst_slices.append(slice(start, start + size))

        out[tuple(dst_slices)] = arr[tuple(src_slices)]
        return out

    def __getitem__(self, idx):
        case = self.case_names[idx]
        case_dir = self.data_dir / case

        imgs = []

        for mod in self.modalities:
            f = case_dir / f"{case}-{mod}.nii.gz"

            if not f.exists():
                raise FileNotFoundError(f"Missing modality file: {f}")

            x = self._load_nii(f)
            x = self._normalize(x)
            x = self._center_crop_or_pad(x, self.patch_size)
            imgs.append(x)

        image = np.stack(imgs, axis=0)

        seg_f = case_dir / f"{case}-seg.nii.gz"

        if not seg_f.exists():
            raise FileNotFoundError(f"Missing segmentation file: {seg_f}")

        seg = self._load_nii(seg_f).astype(np.int64)
        seg = self._center_crop_or_pad(seg, self.patch_size)

        # BraTS GT labels:
        # 0 background, 1 NCR, 2 edema/SNFH, 3 ET

        wt = (seg > 0).astype(np.float32)
        tc = np.logical_or(seg == 1, seg == 3).astype(np.float32)
        et = (seg == 3).astype(np.float32)

        target = np.stack([wt, tc, et], axis=0).astype(np.float32)

        return {
            "case": case,
            "image": torch.from_numpy(image).float(),
            "target": torch.from_numpy(target).float(),
        }


# ============================================================
# nnU-Net loader
# ============================================================

def load_nnunet_model(nnunet_results_dir, device):
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=False,
        perform_everything_on_device=True,
        device=device,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=False,
    )

    model_folder = (
        Path(nnunet_results_dir)
        / "Dataset137_BraTS2021"
        / "nnUNetTrainer__nnUNetPlans__3d_fullres"
    )

    predictor.initialize_from_trained_model_folder(
        str(model_folder),
        use_folds=(5,),
        checkpoint_name="checkpoint_final.pth",
    )

    model = predictor.network.to(device)
    return model


# ============================================================
# Single-fold training
# ============================================================

def train_one_fold(args, fold, split, device):
    print("=" * 80)
    print(f"Training fold {fold}")
    print("=" * 80)

    train_cases = split[str(fold)]["train"]
    val_cases = split[str(fold)]["val"]

    print("Train cases:", len(train_cases))
    print("Val cases:", len(val_cases))

    model = load_nnunet_model(args.nnunet_results, device)
    model.train()

    # Freeze all nnU-Net first
    for p in model.parameters():
        p.requires_grad = False

    # Train only encoder stage 5 and decoder
    for p in model.encoder.stages[5].parameters():
        p.requires_grad = True

    for p in model.decoder.parameters():
        p.requires_grad = True

    prototype_block = PrototypeFusionBlock3D(
        feature_dim=320,
        num_classes=4,
        prototypes_per_class=args.prototypes_per_class,
        num_heads=args.num_heads,
    ).to(device)

    hook_cache = {}

    def prototype_hook(module, inputs, output):
        proto_out = prototype_block(output)

        if "x_fused" in proto_out:
            fused = proto_out["x_fused"]
        elif "fused_features" in proto_out:
            fused = proto_out["fused_features"]
        else:
            raise KeyError(f"Prototype output keys: {proto_out.keys()}")

        hook_cache["proto_similarity"] = proto_out.get("proto_similarity", None)
        hook_cache["class_evidence"] = proto_out.get("class_evidence", None)
        hook_cache["top_proto_indices"] = proto_out.get("top_proto_indices", None)
        hook_cache["fused"] = fused

        return fused

    hook_handle = model.encoder.stages[5].register_forward_hook(prototype_hook)

    train_ds = BraTSPatchDataset(train_cases, args.data_dir)
    val_ds = BraTSPatchDataset(val_cases, args.data_dir)

    train_loader = DataLoader(
        train_ds,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers,
        pin_memory=True,
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=1,
        shuffle=False,
        num_workers=args.num_workers,
        pin_memory=True,
    )

    proto_params = list(prototype_block.parameters())
    nnunet_params = unique_params_from_modules([model.encoder.stages[5], model.decoder])

    print("Trainable prototype params:", sum(p.numel() for p in proto_params))
    print("Trainable nnU-Net params:", sum(p.numel() for p in nnunet_params))

    optimizer = torch.optim.AdamW(
        [
            {"params": proto_params, "lr": args.proto_lr},
            {"params": nnunet_params, "lr": args.nnunet_lr},
        ],
        weight_decay=args.weight_decay,
    )

    criterion = nn.BCEWithLogitsLoss()

    out_dir = Path(args.output_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    checkpoint_path = out_dir / f"prototype_fold{fold}_best.pth"
    results_csv = out_dir / f"prototype_fold{fold}_results.csv"

    best_val_mean = -1.0
    rows = []

    for epoch in range(1, args.epochs + 1):
        model.train()
        prototype_block.train()

        train_losses = []

        for batch in train_loader:
            image = batch["image"].to(device)
            target = batch["target"].to(device)

            optimizer.zero_grad(set_to_none=True)

            logits = model(image)

            loss = criterion(logits, target)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        avg_train_loss = float(np.mean(train_losses))

        model.eval()
        prototype_block.eval()

        val_dices = []

        with torch.no_grad():
            for batch in val_loader:
                image = batch["image"].to(device)
                target = batch["target"].to(device)

                logits = model(image)
                d = compute_region_dice(logits[0], target[0])
                val_dices.append(d)

        wt = float(np.mean([d["WT"] for d in val_dices]))
        tc = float(np.mean([d["TC"] for d in val_dices]))
        et = float(np.mean([d["ET"] for d in val_dices]))
        mean_dice = float(np.mean([d["Mean"] for d in val_dices]))

        print(
            f"Fold {fold} | Epoch {epoch:03d} | "
            f"Loss={avg_train_loss:.4f} | "
            f"WT={wt:.4f} | TC={tc:.4f} | ET={et:.4f} | Mean={mean_dice:.4f}"
        )

        rows.append([epoch, avg_train_loss, wt, tc, et, mean_dice])

        if mean_dice > best_val_mean:
            best_val_mean = mean_dice

            torch.save(
                {
                    "fold": fold,
                    "epoch": epoch,
                    "best_val_mean": best_val_mean,
                    "prototype_block_state_dict": prototype_block.state_dict(),
                    "encoder_stage_5_state_dict": model.encoder.stages[5].state_dict(),
                    "decoder_state_dict": model.decoder.state_dict(),
                    "feature_dim": 320,
                    "num_classes": 4,
                    "prototypes_per_class": args.prototypes_per_class,
                    "insertion_stage": "encoder_stage_5",
                    "training": f"prototype_plus_encoder_stage_5_plus_decoder_fold{fold}",
                    "train_cases": train_cases,
                    "val_cases": val_cases,
                },
                checkpoint_path,
            )

            print("Saved best checkpoint:", checkpoint_path)

    with open(results_csv, "w") as f:
        f.write("epoch,train_loss,WT_Dice,TC_Dice,ET_Dice,Mean_Dice\n")
        for r in rows:
            f.write(",".join(map(str, r)) + "\n")

    if hook_cache:
        print("\nPrototype output shapes from last forward:")
        for k, v in hook_cache.items():
            if torch.is_tensor(v):
                print(k, tuple(v.shape))

    hook_handle.remove()

    print(f"Fold {fold} complete.")
    print("Best validation mean Dice:", best_val_mean)

    return {
        "fold": fold,
        "best_val_mean": best_val_mean,
        "checkpoint": str(checkpoint_path),
        "results_csv": str(results_csv),
    }


# ============================================================
# Summary
# ============================================================

def summarize_results(output_dir):
    output_dir = Path(output_dir)

    rows = []

    for fold in range(5):
        csv_path = output_dir / f"prototype_fold{fold}_results.csv"

        if not csv_path.exists():
            print("Missing CSV:", csv_path)
            continue

        data = np.genfromtxt(csv_path, delimiter=",", names=True)

        if data.ndim == 0:
            best = data
        else:
            best = data[np.argmax(data["Mean_Dice"])]

        rows.append({
            "Fold": fold,
            "Best_Epoch": int(best["epoch"]),
            "WT_Dice": float(best["WT_Dice"]),
            "TC_Dice": float(best["TC_Dice"]),
            "ET_Dice": float(best["ET_Dice"]),
            "Mean_Dice": float(best["Mean_Dice"]),
        })

    summary_csv = output_dir / "prototype_5fold_summary.csv"

    with open(summary_csv, "w") as f:
        f.write("Fold,Best_Epoch,WT_Dice,TC_Dice,ET_Dice,Mean_Dice\n")
        for r in rows:
            f.write(
                f"{r['Fold']},{r['Best_Epoch']},"
                f"{r['WT_Dice']},{r['TC_Dice']},{r['ET_Dice']},{r['Mean_Dice']}\n"
            )

    if len(rows) > 0:
        wt = np.array([r["WT_Dice"] for r in rows])
        tc = np.array([r["TC_Dice"] for r in rows])
        et = np.array([r["ET_Dice"] for r in rows])
        md = np.array([r["Mean_Dice"] for r in rows])

        summary_txt = output_dir / "prototype_5fold_summary.txt"

        with open(summary_txt, "w") as f:
            f.write("Prototype-guided nnU-Net 5-fold summary\n")
            f.write("=======================================\n\n")

            for r in rows:
                f.write(
                    f"Fold {r['Fold']}: "
                    f"WT={r['WT_Dice']:.4f}, "
                    f"TC={r['TC_Dice']:.4f}, "
                    f"ET={r['ET_Dice']:.4f}, "
                    f"Mean={r['Mean_Dice']:.4f}\n"
                )

            f.write("\nAverage across folds:\n")
            f.write(f"WT Dice: {wt.mean():.6f} ± {wt.std(ddof=1):.6f}\n")
            f.write(f"TC Dice: {tc.mean():.6f} ± {tc.std(ddof=1):.6f}\n")
            f.write(f"ET Dice: {et.mean():.6f} ± {et.std(ddof=1):.6f}\n")
            f.write(f"Mean Dice: {md.mean():.6f} ± {md.std(ddof=1):.6f}\n")

        print("Saved summary:", summary_csv)
        print("Saved summary:", summary_txt)


# ============================================================
# Main
# ============================================================

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--data_dir", type=str, required=True)
    parser.add_argument("--nnunet_results", type=str, required=True)
    parser.add_argument("--output_dir", type=str, required=True)

    parser.add_argument("--epochs", type=int, default=15)
    parser.add_argument("--batch_size", type=int, default=1)
    parser.add_argument("--num_workers", type=int, default=2)

    parser.add_argument("--proto_lr", type=float, default=1e-4)
    parser.add_argument("--nnunet_lr", type=float, default=1e-5)
    parser.add_argument("--weight_decay", type=float, default=1e-5)

    parser.add_argument("--prototypes_per_class", type=int, default=8)
    parser.add_argument("--num_heads", type=int, default=4)

    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--fold", type=str, default="all")

    args = parser.parse_args()

    seed_everything(args.seed)

    os.environ["nnUNet_results"] = args.nnunet_results
    os.environ.setdefault("nnUNet_raw", str(Path(args.output_dir) / "nnunet_raw"))
    os.environ.setdefault("nnUNet_preprocessed", str(Path(args.output_dir) / "nnunet_preprocessed"))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    data_dir = Path(args.data_dir)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    case_names = sorted([p.name for p in data_dir.iterdir() if p.is_dir() and p.name.startswith("BraTS-")])

    print("Total cases found:", len(case_names))

    split_json = output_dir / "prototype_5fold_split.json"

    if split_json.exists():
        with open(split_json, "r") as f:
            split = json.load(f)
    else:
        split = create_5fold_split(case_names, split_json, seed=args.seed)

    for fold in range(5):
        print(f"Fold {fold}: train={len(split[str(fold)]['train'])}, val={len(split[str(fold)]['val'])}")

    if args.fold == "all":
        folds_to_run = [0, 1, 2, 3, 4]
    else:
        folds_to_run = [int(args.fold)]

    for fold in folds_to_run:
        train_one_fold(args, fold, split, device)

    summarize_results(output_dir)


if __name__ == "__main__":
    main()